# Qwen3-TTS voice cloning on Kaggle GPU

Free 30 hrs/week of GPU (T4 x2 or P100) with less throttling than free Colab. Nigerian-accented voice cloning via Qwen3-TTS.

## Setup (one-time per notebook)

1. **Enable GPU:** Settings → Accelerator → **GPU T4 x2** (or P100)
2. **Enable internet:** Settings → Internet → **On** (required to install qwen-tts and download model first time)
3. **Upload your reference files as a private dataset:**
   - Click **+ Add Data** (right sidebar) → **New Dataset**
   - Upload `my_voice_full.wav` and `reference_transcript.txt`
   - Name it something like `daniel-voice-ref`
   - Set visibility to **Private**
4. After dataset is attached, it appears at `/kaggle/input/daniel-voice-ref/`
5. Update the `DATASET_NAME` in Cell 3 below if you used a different name.

The model download is cached at `/kaggle/working/hf_cache/` which persists across sessions of this same notebook — first run downloads ~5 GB, subsequent runs load from cache.

## 1. Point HuggingFace cache at persistent Kaggle working dir

In [ ]:
import os
HF_CACHE = '/kaggle/working/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print(f'HuggingFace cache -> {HF_CACHE}')

## 2. Install qwen-tts and patch min_new_tokens

In [ ]:
!pip install -q qwen-tts
print('installed')

import qwen_tts, os
target = os.path.join(os.path.dirname(qwen_tts.__file__), 'core/models/modeling_qwen3_tts.py')
with open(target) as f: src = f.read()
if '"min_new_tokens": 2,' in src:
    src = src.replace('"min_new_tokens": 2,', '"min_new_tokens": kwargs.pop("min_new_tokens", 2),')
    with open(target, 'w') as f: f.write(src)
    print('patched')
else:
    print('already patched or version mismatch')

## 3. Point at the dataset with your reference files

Update `DATASET_NAME` if your Kaggle dataset has a different name. It should contain both `my_voice_full.wav` and `reference_transcript.txt`.

In [ ]:
DATASET_NAME = 'daniel-voice-ref'  # rename to match your Kaggle dataset
REF_AUDIO = f'/kaggle/input/{DATASET_NAME}/my_voice_full.wav'
REF_TEXT_FILE = f'/kaggle/input/{DATASET_NAME}/reference_transcript.txt'

assert os.path.exists(REF_AUDIO), f'Missing: {REF_AUDIO}. Attach the dataset via + Add Data (right sidebar).'
assert os.path.exists(REF_TEXT_FILE), f'Missing: {REF_TEXT_FILE}'

with open(REF_TEXT_FILE) as f: REF_TEXT = f.read().strip()
print(f'Ref audio: {REF_AUDIO}')
print(f'Transcript: {REF_TEXT[:120]}...')

## 4. Load model (first run: ~5 min download; subsequent runs: ~30 sec from cache)

In [ ]:
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base')
print('model loaded')

## 5. Generate (edit TEXT and re-run for different lines)

In [ ]:
import torch, random, numpy as np, soundfile as sf

TEXT = """Hello, my name is Daniel. Let us get into it."""
SEED = 3
TEMPERATURE = 0.7
MIN_TOKENS_PER_CHAR = 10  # lower = faster generation, slight tail-word risk. 15 = safer, slower.
OUT = '/kaggle/working/out.wav'

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
min_new = max(80, len(TEXT) * MIN_TOKENS_PER_CHAR)
print(f'Generating ({len(TEXT)} chars, seed={SEED}, min_new_tokens={min_new})')

wavs, sr = model.generate_voice_clone(
    text=TEXT,
    language='English',
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    temperature=TEMPERATURE,
    min_new_tokens=min_new,
)
wav = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
if hasattr(wav, 'detach'): wav = wav.detach().cpu().numpy()
if wav.ndim == 2 and wav.shape[0] <= 2: wav = wav.T
sf.write(OUT, wav, sr)
print(f'Saved {OUT}: {len(wav)/sr:.1f}s @ {sr}Hz')

from IPython.display import Audio
Audio(OUT)

## 6. Download `out.wav`

Kaggle: click the folder icon in the right sidebar → find `out.wav` under `/kaggle/working/` → click the three dots → **Download**.

Or run the cell below to render inline (also downloadable via the audio widget's menu).

In [ ]:
from IPython.display import Audio
Audio('/kaggle/working/out.wav')